# 🎮 Sistema de Recomendación de Videojuegos de Steam
## Modelo de Machine Learning & MLOps

Este notebook documenta y explica en profundidad la arquitectura, formulación matemática, desarrollo y evaluación del **Motor de Recomendación de Videojuegos para la tienda Steam**, implementado en la API para producción de este proyecto.

---

### 🎯 Objetivos
1. **Comprender el Problema**: Proponer un sistema capaz de sugerir videojuegos relevantes tanto en un esquema **Ítem-Ítem** (juegos similares a uno dado) como **Usuario-Ítem** (juegos acordes al historial de un usuario).
2. **Ingeniería de Características**: Combinar metadatos enriquecidos de videojuegos (géneros, etiquetas de comunidad *tags* y estudio desarrollador).
3. **Modelado con Machine Learning**: Aplicar el modelo vectorial **TF-IDF** (*Term Frequency - Inverse Document Frequency*) y el algoritmo **Nearest Neighbors** con métrica de **Similitud del Coseno**.
4. **Optimización de Producción (MLOps)**: Analizar la eficiencia en memoria (matriz esparsa de 2.3 MB vs densa de 4 GB) y la estrategia de precomputación indexada para respuestas sub-milisegundo en FastAPI.

## 📐 Fundamento Matemático y Teórico

### 1. Modelo de Espacio Vectorial y TF-IDF
Cada videojuego se modela como un documento compuesto por sus atributos descriptivos. La representación vectorial se calcula mediante:

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

Donde:
- $\text{TF}(t, d)$ es la frecuencia del término $t$ en el juego $d$.
- $\text{IDF}(t, D) = \ln\left(\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|}\right) + 1$ penaliza términos excesivamente comunes y realza aquellos distintivos (como mecánicas de juego o géneros de nicho).

### 2. Similitud del Coseno (Cosine Similarity)
Para comparar dos juegos representados por sus vectores $\vec{A}$ y $\vec{B}$, calculamos el coseno del ángulo entre ellos:

$$\text{Cosine Similarity}(\vec{A}, \vec{B}) = \frac{\vec{A} \cdot \vec{B}}{\|\vec{A}\| \|\vec{B}\|} = \frac{\sum_{i=1}^n A_i B_i}{\sqrt{\sum_{i=1}^n A_i^2} \sqrt{\sum_{i=1}^n B_i^2}}$$

- Un valor cercano a **1.0** indica máxima afinidad y congruencia temática.
- Un valor cercano a **0.0** indica independencia temática total.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

# Configuración de visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## 1. Carga y Exploración de los Datos del Catálogo

In [ ]:
# Cargar dataset preprocesado de juegos
games_df = pd.read_parquet('data/processed/games.parquet')
print(f"Total de juegos en catálogo: {len(games_df):,}")
games_df[['id', 'title', 'developer', 'price', 'year', 'genres']].head(5)

### Análisis de Distribución de Géneros Populares
Revisamos cuáles son las categorías más frecuentes en la tienda.

In [ ]:
# Desanidar géneros para contar frecuencias
all_genres = [g for sublist in games_df['genres'] for g in sublist]
genre_counts = pd.Series(all_genres).value_counts()

print("Top 10 géneros con mayor cantidad de títulos:")
display(genre_counts.head(10).to_frame(name='Total Juegos'))

## 2. Ingeniería de Características (Feature Engineering)

Para capturar tanto el estilo del juego como la experiencia de la comunidad y la identidad del creador, construimos una cadena de texto sintética combinando:
1. **Géneros oficiales** (`genres`).
2. **Etiquetas de la comunidad** (`tags`): aportan detalles vitales como *FPS, Multiplayer, Sci-Fi, Pixel Graphics, Open World*.
3. **Desarrollador** (`developer`): los jugadores que disfrutan de un juego suelen valorar otros títulos del mismo estudio.

In [ ]:
# Muestra de la característica combinada para los primeros juegos
for i in range(3):
    row = games_df.iloc[i]
    print(f"🎮 [{row['id']}] {row['title']}:")
    print(f"   Features: {row['features']}\n")

## 3. Vectorización TF-IDF y Eficiencia en Memoria

En entornos de despliegue en la nube (como contenedores en Render o Railway con 512 MB a 1 GB de RAM), una matriz densa de similitud de $22,529 \times 22,529$ ocuparía:

$$22,529 \times 22,529 \times 8 \text{ bytes} \approx 4.06 \text{ GB de RAM}$$

Esto causaría un error catastrófico de **Out-Of-Memory (OOM)**. En su lugar, utilizamos una **matriz esparsa comprimida (CSR)** y un modelo **Nearest Neighbors**, reduciendo el tamaño a unos **2.3 MB**.

In [ ]:
# Vectorización TF-IDF
tfidf = TfidfVectorizer(stop_words='english', max_features=3000)
X_sparse = tfidf.fit_transform(games_df['features'])

mem_bytes = X_sparse.data.nbytes + X_sparse.indices.nbytes + X_sparse.indptr.nbytes
mem_mb = mem_bytes / (1024 * 1024)

print(f"Dimensiones de la matriz esparsa: {X_sparse.shape}")
print(f"Memoria ocupada por la matriz TF-IDF: {mem_mb:.2f} MB")
print(f"Ahorro respecto a una matriz densa: ~99.94% de reducción")

## 4. Entrenamiento del Modelo de Recomendación

Entrenamos el algoritmo `NearestNeighbors` con la métrica de distancia de coseno (`metric='cosine'`). Como la distancia de coseno se define como $1 - \text{Similitud}$, una distancia de $0$ implica identidad y valores pequeños indican alta afinidad.

In [ ]:
nn_model = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')
nn_model.fit(X_sparse)
print("✅ Modelo NearestNeighbors entrenado exitosamente.")

## 5. Demostración del Sistema de Recomendación Ítem-Ítem

Implementamos la función que recibe un `item_id` y devuelve los 5 juegos más recomendados con sus metadatos y puntaje de similitud.

In [ ]:
id_to_idx = {row['id']: i for i, row in games_df.iterrows()}

def recomendar_juego(item_id: str, top_n: int = 5):
    if item_id not in id_to_idx:
        return f"Juego con ID {item_id} no encontrado."
    
    idx = id_to_idx[item_id]
    target_game = games_df.iloc[idx]
    
    # Inferencia de vecinos más cercanos
    dists, indices = nn_model.kneighbors(X_sparse[idx], n_neighbors=top_n + 1)
    
    print(f"🎯 Juego de Entrada: [{target_game['id']}] {target_game['title']}")
    print(f"   Desarrollador: {target_game['developer']} | Precio: ${target_game['price']:.2f}")
    print(f"   Géneros: {', '.join(target_game['genres'])}")
    print("-" * 80)
    
    resultados = []
    for dist, n_idx in zip(dists[0][1:], indices[0][1:]):
        g = games_df.iloc[n_idx]
        similitud = 1.0 - float(dist)
        resultados.append({
            "ID": g['id'],
            "Título": g['title'],
            "Similitud": f"{similitud:.4f}",
            "Desarrollador": g['developer'],
            "Precio": f"${g['price']:.2f}",
            "Géneros": ', '.join(g['genres'])
        })
    
    return pd.DataFrame(resultados)

### Caso de Prueba 1: Juego Indie / Estrategia (`761140` - Lost Summoner Kitty)

In [ ]:
df_recs = recomendar_juego('761140', top_n=5)
display(df_recs)

### Caso de Prueba 2: Búsqueda y Recomendación por Título (ej: Counter-Strike)

In [ ]:
# Búsqueda flexible de ID por nombre
def buscar_por_nombre(nombre: str):
    coincidencias = games_df[games_df['title'].str.contains(nombre, case=False, na=False)]
    if coincidencias.empty:
        return None
    return coincidencias.iloc[0]['id']

cs_id = buscar_por_nombre('Counter-Strike')
if cs_id:
    display(recomendar_juego(cs_id, top_n=5))

## 6. Demostración del Sistema de Recomendación Usuario-Ítem

En este enfoque, analizamos las reseñas del usuario para extraer los juegos que recomendó explícitamente (`recommend == True`). Agregamos los juegos afines y filtramos aquellos que el usuario ya conoce, recomendándole títulos novedosos con alta afinidad.

In [ ]:
reviews_df = pd.read_parquet('data/processed/reviews.parquet')

def recomendar_para_usuario(user_id: str, top_n: int = 5):
    user_revs = reviews_df[(reviews_df['user_id'] == user_id) & (reviews_df['recommend'] == True)]
    if user_revs.empty:
        user_revs = reviews_df[reviews_df['user_id'] == user_id]
    
    if user_revs.empty:
        return f"Usuario {user_id} no encontrado o sin reseñas."
    
    user_games = user_revs['item_id'].unique().tolist()
    print(f"👤 Usuario: {user_id}")
    print(f"🎮 Juegos evaluados positivamente por el usuario: {user_games}")
    
    candidatos = {}
    for g_id in user_games:
        if g_id in id_to_idx:
            idx = id_to_idx[g_id]
            dists, indices = nn_model.kneighbors(X_sparse[idx], n_neighbors=top_n + 1)
            for dist, n_idx in zip(dists[0][1:], indices[0][1:]):
                rec_g = games_df.iloc[n_idx]
                rec_id = rec_g['id']
                if rec_id not in user_games:
                    sim = 1.0 - float(dist)
                    if rec_id not in candidatos or sim > candidatos[rec_id]['similitud']:
                        candidatos[rec_id] = {
                            "ID": rec_id,
                            "Título": rec_g['title'],
                            "Similitud": f"{sim:.4f}",
                            "similitud_num": sim,
                            "Desarrollador": rec_g['developer'],
                            "Géneros": ', '.join(rec_g['genres'])
                        }
    
    ordenados = sorted(candidatos.values(), key=lambda x: x['similitud_num'], reverse=True)[:top_n]
    res_df = pd.DataFrame(ordenados).drop(columns=['similitud_num'])
    return res_df

# Prueba con usuario activo del dataset
display(recomendar_para_usuario('76561197970982479', top_n=5))

## 7. Despliegue en Producción & Consideraciones MLOps

### Estrategia Híbrida de Inferencia:
1. **Precomputación Indexada**: Durante la fase de ETL (`scripts/build_dataset.py`), precomputamos el Top-5 de juegos similares para todos los títulos del catálogo y los guardamos en `data/processed/recommendations.json`. Esto permite que la API en FastAPI responda en **tiempo constante $O(1)$** (< 5 ms).
2. **Fallback Dinámico**: Para juegos nuevos o consultas dinámicas, la API mantiene cargado el modelo serializado `recommendation_model.joblib` en memoria mediante el ciclo de vida `lifespan` de FastAPI, permitiendo vectorizar e inferir en tiempo real sin reiniciar el servicio.

### Conclusiones:
- Se logró un MVP funcional, testeado y reproducible que cumple al 100% las consignas de Henry y los requerimientos de producción.
- La arquitectura desacoplada (`ETL -> Artefactos -> FastAPI DataService -> Endpoints`) asegura alta escalabilidad y bajo consumo de recursos.